# Smartphone Addiction: validated Python baseline

A reproducible histogram gradient boosting baseline with explicit schema checks.

In [ ]:
import os
from glob import glob
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

train_paths = glob('/kaggle/input/**/train.csv', recursive=True)
assert len(train_paths) == 1, f'Expected one attached train.csv, found: {train_paths}'
data_dir = os.path.dirname(train_paths[0])
train = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test = pd.read_csv(os.path.join(data_dir, 'test.csv'))
sample = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))
target, id_column = 'addicted_label', 'id'
features = [c for c in train.columns if c not in {target, id_column}]
X, y, X_test = train[features], train[target], test[features]
categorical = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric = [c for c in features if c not in categorical]
preprocess = ColumnTransformer([
    ('numeric', SimpleImputer(strategy='median'), numeric),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ]), categorical),
])
model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', HistGradientBoostingClassifier(learning_rate=0.08, max_iter=120, max_leaf_nodes=31, l2_regularization=1.0, random_state=81)),
])
model.fit(X, y)
probabilities = model.predict_proba(X_test)[:, 1]
submission = sample.copy()
submission[target] = probabilities
assert list(submission.columns) == [id_column, target]
assert len(submission) == len(test)
assert submission[id_column].equals(test[id_column])
assert np.isfinite(probabilities).all()
assert ((probabilities >= 0) & (probabilities <= 1)).all()
submission.to_csv('/kaggle/working/submission.csv', index=False)
print({'train_rows': len(train), 'test_rows': len(test), 'features': len(features), 'probability_mean': float(probabilities.mean())})
submission.head()